# Visualize Preprocessed DNS Output

This notebook loads a subsampled `.npz` file (generated by `preprocess_dns_output.py`) and provides interactive tools to visualize the data. It contains two sections:

1.  **Time Evolution Plot:** A time-vs-z plot (Hovmöller diagram) to see the temporal evolution at a single spatial point.
2.  **Spatial Slice Plot:** An x-vs-z plot to see the spatial structure at a single moment in time.

### 1. Setup and Imports

In [ ]:
from pathlib import Path
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Make sure your packages are installed in editable mode
from mhd_surrogate_core.plot import plot_z_time_evolution, plot_xz_slice

### 2. Load Data and Configure Widgets

**Action Required:** Change the `data_file_path` variable below to point to the specific `.npz` file you want to analyze.

In [ ]:
# Example path - update this to your specific file
data_file_path = Path("/cephfs/users/skowronek/Documents/PhD/nuclear_fusion_cooling/data/preprocessed_dns_output/01-Cold_Runs/01-Re16K_Ha325/interp/interp.npz")

# Load the data once to configure the interactive widgets
if not data_file_path.exists():
    print(f"ERROR: Data file not found at {data_file_path}")
    # Create dummy data for widget initialization if file not found
    channel_options = ['vx']
    max_x_index = 10
    max_y_index = 10
    max_time_index = 10
else:
    with np.load(data_file_path, allow_pickle=True) as data:
        channel_options = list(data['labels'])
        max_time_index = data['timeseries'].shape[0] - 1
        max_x_index = data['timeseries'].shape[1] - 1
        max_y_index = data['timeseries'].shape[2] - 1

---

### 3. Time Evolution Plot (Time vs. Z)

In [ ]:
channel_dropdown_t = widgets.Dropdown(options=channel_options, description='Channel:', value=channel_options[0])
x_slider_t = widgets.IntSlider(min=0, max=max_x_index, value=max_x_index // 2, description='X Index:')
y_slider_t = widgets.IntSlider(min=0, max=max_y_index, value=max_y_index // 2, description='Y Index:')

interactive_plot_t = widgets.interactive_output(
    plot_z_time_evolution,
    {'data_path': widgets.fixed(data_file_path), 
     'channel': channel_dropdown_t, 
     'x_index': x_slider_t, 
     'y_index': y_slider_t}
)

display(widgets.VBox([widgets.HBox([channel_dropdown_t, x_slider_t, y_slider_t]), interactive_plot_t]))

---

### 4. Spatial Slice Plot (X vs. Z)

In [ ]:
channel_dropdown_s = widgets.Dropdown(options=channel_options, description='Channel:', value=channel_options[0])
y_slider_s = widgets.IntSlider(min=0, max=max_y_index, value=max_y_index // 2, description='Y Index:')
time_slider_s = widgets.IntSlider(min=0, max=max_time_index, value=max_time_index // 2, description='Time Index:')

interactive_plot_s = widgets.interactive_output(
    plot_xz_slice,
    {'data_path': widgets.fixed(data_file_path), 
     'channel': channel_dropdown_s, 
     'y_index': y_slider_s, 
     'time_index': time_slider_s}
)

display(widgets.VBox([widgets.HBox([channel_dropdown_s, y_slider_s, time_slider_s]), interactive_plot_s]))